In [1]:
"""
Generate one horizontal-bar TF-IDF chart per subreddit, styled to match the
reference image: teal rounded header banner (title + italic subtitle),
clean bars with value labels, no axis clutter.

Usage:
    python make_tfidf_charts.py
Reads:  03_tfidf_top10_per_subreddit.csv  (columns: subreddit, rank, term, tfidf)
Writes: out/tfidf_<subreddit>.png  (one file per subreddit)
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# ── Config ──────────────────────────────────────────────────────────────────
CSV_PATH   = "../PROCESSED/03_tfidf_top10_per_subreddit.csv"
OUT_DIR    = "../PROCESSED/03_tfidf_charts"
BAR_COLOR  = "#0E6E63"        # teal, matches reference
BG_COLOR   = "#F7F8F7"        # light off-white background
LABEL_COLOR = "#3A4A4A"
VALUE_COLOR = "#6B7B7B"
HEADER_TEXT_COLOR = "#FFFFFF"

# Optional: human-readable subtitle per subreddit (shown under the title).
# Falls back to "r/<name>" / no subtitle if not listed here.
SUBREDDIT_LABELS = {
    "climatechange": ("r/climatechange", "climate-specific"),
    "climate":       ("r/climate",       "climate-specific"),
    "energy":        ("r/energy",        "energy & resources"),
    "europe":        ("r/europe",        "regional / policy"),
    "news":          ("r/news",          "general-news"),
    "politics":      ("r/politics",      "general-news"),
    "science":       ("r/science",       "science-focused"),
    "worldnews":     ("r/worldnews",     "general-news"),
    "Futurology":    ("r/Futurology",    "future / tech"),
    "changemyview":  ("r/changemyview",  "debate / opinion"),
}


def make_chart(df_sub: pd.DataFrame, subreddit: str, out_path: str):
    title, subtitle = SUBREDDIT_LABELS.get(subreddit, (f"r/{subreddit}", ""))

    # Sort so the #1 term appears at the TOP of the chart
    df_sub = df_sub.sort_values("rank")
    terms  = df_sub["term"].tolist()[::-1]
    vals   = df_sub["tfidf"].tolist()[::-1]

    fig_h = 6.2
    fig, ax = plt.subplots(figsize=(5.0, fig_h))
    fig.patch.set_facecolor(BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    bars = ax.barh(terms, vals, color=BAR_COLOR, height=0.62, zorder=3)

    # Value labels just past the bar end
    max_val = max(vals)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_width() + max_val * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.2f}",
            va="center", ha="left", fontsize=11, color=VALUE_COLOR,
        )

    # Strip all chart chrome — axis-free look like the reference
    ax.set_xlim(0, max_val * 1.22)
    ax.set_xticks([])
    for spine in ["top", "right", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#D9DDDB")
    ax.tick_params(axis="y", length=0, labelsize=12, colors=LABEL_COLOR)
    ax.tick_params(axis="x", length=0)

    plt.subplots_adjust(left=0.30, right=0.95, top=0.80, bottom=0.04)

    # ── Header banner (rounded teal box with title + subtitle) ──
    # Drawn in figure coordinates so it sits above the axes as its own block.
    header_ax = fig.add_axes([0.04, 0.83, 0.92, 0.14])
    header_ax.set_xlim(0, 1); header_ax.set_ylim(0, 1)
    header_ax.axis("off")
    header_ax.add_patch(FancyBboxPatch(
        (0, 0), 1, 1,
        boxstyle="round,pad=0,rounding_size=0.18",
        linewidth=0, facecolor=BAR_COLOR, transform=header_ax.transAxes,
        clip_on=False,
    ))
    if subtitle:
        header_ax.text(0.5, 0.62, title, ha="center", va="center",
                        fontsize=15, fontweight="bold", color=HEADER_TEXT_COLOR,
                        transform=header_ax.transAxes)
        header_ax.text(0.5, 0.22, subtitle, ha="center", va="center",
                        fontsize=10.5, style="italic", color=HEADER_TEXT_COLOR,
                        transform=header_ax.transAxes)
    else:
        header_ax.text(0.5, 0.5, title, ha="center", va="center",
                        fontsize=15, fontweight="bold", color=HEADER_TEXT_COLOR,
                        transform=header_ax.transAxes)

    fig.savefig(out_path, dpi=200, facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  Saved -> {out_path}")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    df = pd.read_csv(CSV_PATH)

    for subreddit, df_sub in df.groupby("subreddit", sort=False):
        out_path = os.path.join(OUT_DIR, f"tfidf_{subreddit}.png")
        make_chart(df_sub, subreddit, out_path)

    print(f"\nDone. {df['subreddit'].nunique()} charts written to '{OUT_DIR}/'.")


if __name__ == "__main__":
    main()

  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_Futurology.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_changemyview.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_climate.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_climatechange.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_energy.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_europe.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_news.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_politics.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_science.png
  Saved -> ../PROCESSED/03_tfidf_charts\tfidf_worldnews.png

Done. 10 charts written to '../PROCESSED/03_tfidf_charts/'.


In [3]:
"""
Generate a horizontal bar chart of controversy rate by subreddit, styled to
match the reference image: dark navy background, ascending bars (lowest at
top, highest at bottom), with the top-N highest bars highlighted in coral/
orange and the rest in muted teal.

Usage:
    python make_controversy_chart.py
Reads:  08_controversiality_by_subreddit.csv  (columns: subreddit, controversy_rate, n_controversial, n_total)
Writes: 08_controversy_rate_by_subreddit.png
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Config ──────────────────────────────────────────────────────────────────
CSV_PATH    = "../PROCESSED/08_controversiality_by_subreddit.csv"
OUT_PATH    = "../PROCESSED/08_controversy_rate_by_subreddit.png"

#BG_COLOR     = "#1B2A33"   # dark navy background
BG_COLOR     = "#F7F8F7"   # light off-white background
TEAL_LIGHT   = "#5FA89E"   # muted teal for most bars
CORAL        = "#E0623D"   # accent color for the highest bars
TEXT_COLOR   = "#1B2A33"   # dark navy text
GRID_COLOR   = "#33454A"

HIGHLIGHT_TOP_N = 3        # how many highest-rate subreddits to highlight in coral
TITLE = "Controversy Rate by Subreddit"


def main():
    df = pd.read_csv(CSV_PATH)
    df["controversy_pct"] = df["controversy_rate"] * 100

    # Ascending sort -> lowest rate at TOP of chart, highest at BOTTOM
    # (matplotlib's barh draws the first row at the bottom, so to put the
    # lowest value at the top we sort descending and let barh's natural
    # bottom-up order place it correctly)
    df = df.sort_values("controversy_pct", ascending=True)

    subs = df["subreddit"].tolist()
    vals = df["controversy_pct"].tolist()

    # Highlight the top-N highest values in coral, rest in teal
    threshold = sorted(vals)[-HIGHLIGHT_TOP_N]
    colors = [CORAL if v >= threshold else TEAL_LIGHT for v in vals]

    fig, ax = plt.subplots(figsize=(5.0, 4.0))
    fig.patch.set_facecolor(BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    bars = ax.barh(subs, vals, color=colors, height=0.62, zorder=3)

    # Value labels just past each bar's end
    max_val = max(vals)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_width() + max_val * 0.03,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.1f}%",
            va="center", ha="left", fontsize=9.5, color=TEXT_COLOR,
        )

    # Title
    ax.set_title(TITLE, color=TEXT_COLOR, fontsize=12, pad=12, loc="center")

    # Axis styling — light gridlines, no spines, light tick labels
    ax.set_xlim(0, max_val * 1.18)
    ax.tick_params(axis="y", length=0, labelsize=9.5, colors=TEXT_COLOR)
    ax.tick_params(axis="x", length=0, labelsize=8.5, colors="#9AA8A6")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    fig.savefig(OUT_PATH, dpi=200, facecolor=BG_COLOR)
    plt.close(fig)
    print(f"Saved -> {OUT_PATH}")


if __name__ == "__main__":
    main()

Saved -> ../PROCESSED/08_controversy_rate_by_subreddit.png


In [11]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Config ──────────────────────────────────────────────────────────────────
CSV_PATH    = "../PROCESSED/00_subreddit_summary.csv"
OUT_PATH    = "../PROCESSED/00_comments_per_subreddit.png"

#BG_COLOR     = "#1B2A33"   # dark navy background
BG_COLOR     = "#F7F8F7"   # light off-white background
TEAL_LIGHT   = "#5FA89E"   # muted teal for most bars
CORAL        = "#E0623D"   # accent color for the highest bars
TEXT_COLOR   = "#1B2A33"   # dark navy text
GRID_COLOR   = "#33454A"

#HIGHLIGHT_TOP_N = 3        # how many highest-rate subreddits to highlight in coral
TITLE = "Comments per Subreddit"


def main():
    df = pd.read_csv(CSV_PATH)
    df["comments_pct"] = df["n_comments"] 

    # Ascending sort -> lowest rate at TOP of chart, highest at BOTTOM
    # (matplotlib's barh draws the first row at the bottom, so to put the
    # lowest value at the top we sort descending and let barh's natural
    # bottom-up order place it correctly)
    df = df.sort_values("comments_pct")#, ascending=False)

    subs = df["subreddit"].tolist()
    vals = df["comments_pct"].tolist()

    # Highlight the top-N highest values in coral, rest in teal
    #threshold = sorted(vals)[-HIGHLIGHT_TOP_N]
    #colors = [CORAL if v >= threshold else TEAL_LIGHT for v in vals]

    fig, ax = plt.subplots(figsize=(5.0, 4.0))
    fig.patch.set_facecolor(BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    bars = ax.barh(subs, vals, color=TEAL_LIGHT, height=0.62, zorder=3)

    # Value labels just past each bar's end
    max_val = max(vals)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_width() + max_val * 0.03,
            bar.get_y() + bar.get_height() / 2,
            f"{v}",
            va="center", ha="left", fontsize=9.5, color=TEXT_COLOR,
        )

    # Title
    ax.set_title(TITLE, color=TEXT_COLOR, fontsize=12, pad=12, loc="center")

    # Axis styling — light gridlines, no spines, light tick labels
    ax.set_xlim(0, max_val * 1.18)
    ax.tick_params(axis="y", length=0, labelsize=9.5, colors=TEXT_COLOR)
    ax.tick_params(axis="x", length=0, labelsize=8.5, colors="#9AA8A6")
    for spine in ax.spines.values():
        spine.set_visible(False)
    #ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    #ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=True))

    plt.tight_layout()
    fig.savefig(OUT_PATH, dpi=300, facecolor=BG_COLOR)
    plt.close(fig)
    print(f"Saved -> {OUT_PATH}")


if __name__ == "__main__":
    main()

Saved -> ../PROCESSED/00_comments_per_subreddit.png
